# 02. 데이터 전처리와 MySQL ETL

원본 설문에 필요한 정제 기준을 적용하고 MySQL에 적재한다. 03은 이 단계에서 만든 SQL 뷰를 조회한다.

## 1. 원본 설문 불러오기

원본은 269명·20개 열이다. 추첨용 이메일은 분석 데이터에서 제외한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if ((candidate / 'notebooks').is_dir()
                and (candidate / 'sql').is_dir()
                and (candidate / 'data' / 'q18_audited_labels.csv').is_file()):
            return candidate
    raise FileNotFoundError(f'프로젝트 루트를 찾지 못했습니다: {start}')

ROOT = find_project_root()
RAW = ROOT / 'raw_data' / '패션_플랫폼_설문_통합.csv'
if not RAW.exists():
    raise FileNotFoundError('프로젝트 루트와 원본 설문 CSV를 확인하세요.')

df = pd.read_csv(RAW, encoding='utf-8-sig')
raw_n = len(df)
print(f'원본 설문: {raw_n}명, {df.shape[1]}개 열')
assert df.shape == (269, 20)

원본 설문: 269명, 20개 열


## 2. 필요한 컬럼과 Q18 무응답 정리

모순 판정·데이터 개요·최종 분석에 쓰는 문항만 남긴다. Q6은 이후 플랫폼별·복수 플랫폼 집계를 위해 플랫폼명을 표준화한다. Q16은 이 단계에서 원문을 유지한다.

In [2]:
RENAME = {
    '타임스탬프': 'timestamp',
    'Q1. 성별을 선택해주세요.': 'gender',
    'Q2. 연령대를 선택해주세요.': 'age',
    'Q5. 패션 플랫폼을 사용하시나요? (아래 질문에서 "아니오"를 선택하시면 마지막 질문으로 바로 이동합니다.)': 'uses_platform',
    'Q6. 현재 가장 자주 사용하는 패션 관련 플랫폼을 1~2개 선택해주세요.': 'platforms',
    'Q7. 패션 플랫폼을 선택할 때 가장 중요하게 보는 요소는 무엇인가요? (최대 3개 선택)': 'selection_factors',
    'Q8. 주로 어떤 목적으로 패션 플랫폼을 열어보나요? (최대 2개 선택)': 'open_purpose',
    'Q9. 최근 6개월 동안 패션 플랫폼에서 구매한 횟수는 몇 번인가요?': 'purchase_count',
    'Q10. 패션 플랫폼에서 가장 최근에 구매한 시점은 언제인가요?': 'last_purchase',
    'Q11. 패션 플랫폼에서 한 번 구매할 때 평균 지출 금액은 얼마인가요?': 'avg_spend',
    'Q12. 같은 플랫폼에서 다시 구매하게 된 가장 큰 이유는 무엇인가요? (최대 2개 선택)': 'repurchase_reason',
    'Q13. 패션 플랫폼에서 구매한 경험이 있다면, 불만족스러웠던 경험이 있나요? (복수 선택)': 'dissatisfaction',
    'Q14. 지금 가장 자주 사용하는 플랫폼을 앞으로도 계속 사용할 것 같나요?': 'continue_use',
    'Q15. 현재 사용하는 패션 플랫폼을 지인에게 추천할 의향이 얼마나 있나요?': 'nps',
    'Q16. 지금 사용하는 패션 앱을 처음 알게 된 경로는 무엇인가요?': 'discovery',
    'Q17. 최근 6개월 내 패션 상품 구매에 가장 영향을 준 채널은 무엇인가요?': 'influence',
    'Q18. (선택) 패션 앱을 사용하면서 아쉬웠던 점이 있다면 자유롭게 적어주세요.': 'feedback',
}
df = df.rename(columns=RENAME)[list(RENAME.values())]
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['nps'] = df['nps'].astype(pd.Int64Dtype())

Q6 플랫폼 이름을 아래 코드에서 표준화한다. 선택지 내부 쉼표와 복합 자유입력을 먼저 처리해 이후 복수응답 집계의 기준을 맞춘다.

In [3]:
# 긴 선택지 원문과 임시 placeholder 정의
# 내부에 쉼표가 있어서 split 전에 미리 치환해둬야 함
LONG_OPTION = '종합 쇼핑몰 (쿠팡, 네이버 쇼핑, 테무, 알리익스프레스 등)'
PLACEHOLDER = '__종합쇼핑몰__'

# 공백으로 붙여 쓴 복합 항목 → 두 개의 플랫폼으로 분리 (1:2 매핑)
SPACE_SPLIT = {
    '자라 룩핀': ['자라', '룩핀'],
}

# 자유 입력 및 표기 변형 → 표준명으로 통일 (1:1 매핑)
PLATFORM_MAP = {
    PLACEHOLDER: '종합 쇼핑몰',   # placeholder → 표준명 복원
    'LOOKPIN': '룩핀',             # 영문 표기 → 한글 통일
    'Shein': '쉬인',               # 영문 표기 → 한글 통일
    'CIDER': '기타',               # 인식 불가 플랫폼 → 기타
    '뉴발 나이키 등': '기타',
    '브랜드 온라인 매장 (사이트)': '기타',
    '에잇세컨즈': '기타',
    '인스타 자사몰': '기타',
    '오까네': '기타',
    '해외어플': '기타',
    '하바티': '기타',
    '아이엠샵': '기타',
    '나인제트': '기타',
    '기타': '기타',
}

def clean_platforms(val):
    # 결측값이면 그대로 반환
    if pd.isna(val):
        return val
    
    # 쉼표 포함된 긴 선택지를 placeholder로 치환 (split 오류 방지)
    s = val.replace(LONG_OPTION, PLACEHOLDER)
    
    # 쉼표 기준으로 분리 후 앞뒤 공백 제거
    items = [x.strip() for x in s.split(',')]
    
    result = []
    for item in items:
        # 공백으로 붙여 쓴 복합 항목이면 두 개로 분리
        if item in SPACE_SPLIT:
            result.extend(SPACE_SPLIT[item])
        # 나머지는 표준명으로 변환 (매핑 없으면 원본 유지)
        else:
            result.append(PLATFORM_MAP.get(item, item))
    
    # 중복 제거 (순서 유지)
    seen = set()
    deduped = [x for x in result if not (x in seen or seen.add(x))]
    
    # 다시 쉼표로 합쳐서 반환
    return ', '.join(deduped)

# 전체 platforms 컬럼에 정제 함수 적용
df['platforms'] = df['platforms'].apply(clean_platforms)

In [4]:
NULL_LIKE = {
    '없음', '없어요', '없습니다', '없어용ㅎㅎ', '없어요.', '없습니다.',
    '없음.', '없어요!', '없어.', '없음 ㅎ', '없어요 ㅎ',
    '-', 'none', '딱히', '딱히.',
    '좋아요', 'nowadays fashion apps are so easy to use',
    '딱히 큰 불편함 없이 사용해옴',
    '한 번도 사용할 적은 없지만 굳이 뽑자면.... 없습니다',
}

df['feedback'] = df['feedback'].apply(
    lambda x: np.nan if pd.notna(x) and str(x).strip().lower() in NULL_LIKE else x
)

## 3. 논리 모순 제거

설문 분기에서 발생할 수 있는 논리 모순 6종을 확인하고, 남은 응답에 `user_id`를 부여한다.

In [5]:
# 모순 1: Q5 비사용자인데 Q6~Q17 중 응답이 있는 경우
m1 = (
    (df['uses_platform'] == '아니오') &
    (df['platforms'].notna() | df['nps'].notna() | df['discovery'].notna())
)

# 모순 2: Q9 비구매자인데 Q10~Q12 중 응답이 있는 경우
m2 = (
    (df['purchase_count'] == '구매하지 않음') &
    (df['last_purchase'].notna() | df['avg_spend'].notna() | df['repurchase_reason'].notna())
)

# 모순 3: Q9 구매자인데 Q10~Q12 중 빠진 응답이 있는 경우
m3 = (
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음') &
    (df['last_purchase'].isna() | df['avg_spend'].isna() | df['repurchase_reason'].isna())
)

# 모순 4: Q13 "구매 경험 자체가 없음"인데 Q9에 구매 횟수가 있는 경우
m4 = (
    df['dissatisfaction'].str.contains('구매 경험 자체가 없음', na=False) &
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음')
)

# 모순 5: Q17 "최근 6개월 내 구매 경험 없음"인데 Q9에 구매 횟수가 있는 경우
m5 = (
    (df['influence'] == '최근 6개월 내 패션 상품 구매 경험 없음') &
    df['purchase_count'].notna() &
    (df['purchase_count'] != '구매하지 않음')
)

# 모순 6: Q13 "불만족 경험 없음"과 실제 불만 항목이 함께 선택된 경우
dissatisfaction_items = df['dissatisfaction'].fillna('').str.split(', ')
m6 = dissatisfaction_items.apply(
    lambda items: ('불만족 경험 없음' in items) and (len([x for x in items if x.strip()]) > 1)
)

inconsistent_mask = m1 | m2 | m3 | m4 | m5 | m6
for i, m in enumerate([m1, m2, m3, m4, m5, m6], 1):
    print(f'모순 {i}: {m.sum()}건')
print(f'총 제거: {inconsistent_mask.sum()}건')
df = df[~inconsistent_mask].reset_index(drop=True)

# user_id 부여 (timestamp 정렬 후 1부터)
df = df.sort_values('timestamp').reset_index(drop=True)
df.insert(0, 'user_id', range(1, len(df) + 1))

clean_n = len(df)
assert (raw_n, clean_n) == (269, 266)
print(f'분석 가능 응답: {raw_n} → {clean_n}명')

모순 1: 0건
모순 2: 0건
모순 3: 0건
모순 4: 2건
모순 5: 0건
모순 6: 1건
총 제거: 3건
분석 가능 응답: 269 → 266명


## 4. 정제 응답을 MySQL에 적재

`to_sql`로 정제 응답 266명을 `final_survey`에 적재하고 기본키를 지정한다. `survey` 테이블은 보존한다.

In [6]:
import os
import re
from dotenv import load_dotenv
from sqlalchemy import URL, Integer, String, create_engine, text

load_dotenv(ROOT / '.env')
required = ('DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_NAME')
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise RuntimeError(f'MySQL 환경변수 누락: {", ".join(missing)}')
DB_NAME = os.environ['DB_NAME']
if not re.fullmatch(r'[A-Za-z0-9_]+', DB_NAME):
    raise ValueError('DB_NAME은 영문·숫자·밑줄만 사용할 수 있습니다.')
root_url = URL.create(
    'mysql+mysqlconnector', username=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'], host=os.environ['DB_HOST'],
)
with create_engine(root_url).begin() as conn:
    conn.exec_driver_sql(
        f'CREATE DATABASE IF NOT EXISTS `{DB_NAME}` '
        'CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'
    )
engine = create_engine(root_url.set(database=DB_NAME), pool_pre_ping=True)
print('MySQL 연결 및 최종 분석 전용 테이블 준비 완료')

MySQL 연결 및 최종 분석 전용 테이블 준비 완료


In [7]:
audit = pd.DataFrame([{
    'raw_n': raw_n,
    'removed_n': int(inconsistent_mask.sum()),
    'cleaned_n': clean_n,
}])
with engine.begin() as conn:
    df.to_sql('final_survey', con=conn, if_exists='replace', index=False,
              dtype={'user_id': Integer()})
    conn.exec_driver_sql('ALTER TABLE final_survey MODIFY user_id INT NOT NULL PRIMARY KEY')
    audit.to_sql('final_etl_audit', con=conn, if_exists='replace', index=False)
print(f'final_survey 적재: {len(df)}명 (원본 survey 보존)')

final_survey 적재: 266명 (원본 survey 보존)


## 5. Q18 확정 라벨 적재

Q18 자유응답을 전수 검토해 확정한 다중 라벨을 `data/q18_audited_labels.csv`에서 읽는다. 유효 응답과 안정 키로 1:1 병합하고, 전체 매칭·중복·라벨 동일성을 확인한 뒤 MySQL에 적재한다.

In [8]:
import hashlib

# 타임스탬프는 원본 269건에서 유일하다. 응답 내용까지 결합해
# 행 순서가 바뀌거나 내용이 달라져도 잘못된 라벨을 조용히 붙이지 않는다.
assert df['timestamp'].is_unique
q18_valid = df.loc[
    df['feedback'].notna()
    & df['feedback'].ne('패션 앱을 자주 사용하지 않음'),
    ['user_id', 'timestamp', 'feedback', 'uses_platform']
].copy()
assert len(q18_valid) == 92
assert q18_valid['uses_platform'].eq('예').sum() == 68

def response_key(row):
    payload = f"{row['timestamp'].isoformat()}\x1f{row['feedback']}"
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

q18_valid['response_key'] = q18_valid.apply(response_key, axis=1)
labels = pd.read_csv(ROOT / 'data' / 'q18_audited_labels.csv',
                     encoding='utf-8-sig')
assert len(labels) == labels['response_key'].nunique() == 92
assert q18_valid['response_key'].is_unique
assert labels['categories'].notna().all()
assert labels['response_key'].str.fullmatch(r'[0-9a-f]{64}').all()

matched = q18_valid.merge(labels, on='response_key', how='left',
                          validate='one_to_one', indicator=True)
assert len(matched) == 92
assert matched['_merge'].eq('both').all(), '라벨 미매칭 응답이 있습니다.'
assert matched['categories'].notna().all()

# 확정 라벨 파일 92건의 변경 여부를 확인하는 매핑 지문.
canonical = '\n'.join(
    f'{r.response_key}:{r.categories}'
    for r in matched[['response_key', 'categories']]
        .sort_values('response_key').itertuples(index=False)
)
EXPECTED_LABEL_SHA256 = '77ee610e4190c532ac2165b3bb4716ebd248f3adaa355cea01938c48a0ddd8f8'
assert hashlib.sha256(canonical.encode('utf-8')).hexdigest() == EXPECTED_LABEL_SHA256

label_rows = matched[['user_id', 'categories']].copy()
label_rows['category'] = label_rows['categories'].str.split('|')
label_rows = label_rows.explode('category')[['user_id', 'category']]
assert not label_rows.duplicated().any()
assert label_rows['user_id'].nunique() == 92
with engine.begin() as conn:
    label_rows.to_sql('final_q18_labels', con=conn, if_exists='replace',
                      index=False, dtype={'user_id': Integer(), 'category': String(40)})
    conn.exec_driver_sql(
        'ALTER TABLE final_q18_labels ADD PRIMARY KEY (user_id, category)'
    )
print('Q18 확정 라벨: 92건 매칭 · 미매칭 0건 · 중복 키 0건')

Q18 확정 라벨: 92건 매칭 · 미매칭 0건 · 중복 키 0건


## 6. 최종 분석용 SQL 뷰와 분모 확인

분석에 필요한 모수·점수 정의를 [02 SQL](../sql/02_data_preparation.sql)에 둔다. 뷰를 생성한 뒤 원본→정제→가설별 집단의 분모를 확인한다.

In [9]:
def load_queries(path):
    body = Path(path).read_text(encoding='utf-8')
    parts = re.split(r'(?m)^--\s*name:\s*(\w+).*$', body)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

Q = load_queries(ROOT / 'sql' / '02_data_preparation.sql')
with engine.begin() as conn:
    conn.exec_driver_sql(Q['semantic_view'])
counts = pd.read_sql(Q['population'], engine).iloc[0]
expected = {
    'raw': 269, 'cleaned': 266, 'platform_users': 200,
    'buyers': 191, 'valid_q18': 92, 'platform_user_q18': 68,
    'labeled_q18': 92,
}
assert counts.to_dict() == expected, counts.to_dict()
display(counts.to_frame(name='응답자 수'))

,응답자 수
raw,269
cleaned,266
platform_users,200
buyers,191
valid_q18,92
platform_user_q18,68
labeled_q18,92
